In [1]:
# Install needed libraries
%pip install -U python-jobspy
%pip install tqdm
%pip install xlsxwriter
%pip install tenacity requests

# Install MongoDB Python driver
%pip install pymongo
%pip install python-dotenv

You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upg

In [2]:
import sys
from pathlib import Path

# Get the absolute path of the project root (one level up from the notebooks directory)
project_root = str(Path().resolve().parent)  # Goes up two levels to reach the project root

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from operations import (
    process_and_save_jobs, 
    setup_output_directory, 
    connect_to_mongodb,
    hours_old_since_2025,
    safe_scrape_jobs
)
import itertools

In [4]:
connect_to_mongodb()

Looking for .env at: /home/wagner/Documentos/dev-projects/No Country/Market-Scraper/.env
✅ Successfully connected to MongoDB
📊 Database: job_market
📂 Collection: jobs
🔗 Total documents: 14875


{'client': MongoClient(host=['ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000),
 'collection': Collection(Database(MongoClient(host=['ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000), 'job_market'), 'jobs')}

In [5]:
# --- 1. Definir Directorio de Salida ---
output_dir = setup_output_directory("../data/raw")
print(f"Directorio de salida: {output_dir}")

Directorio de salida: ../data/raw/jobs_20251127_205022


## 2. Definir Parámetros de Búsqueda Base

In [6]:
sectores_clave = ["Fintech", "EdTech", "Future of Work"]
search_terms = sectores_clave
hours_old= hours_old_since_2025(2025)
hours_old_list = list(range(0, hours_old, 24))
indeed_glassdoor_countries = [
    "Australia",
    "Austria",
    "Belgium",
    "Brazil",
    "Canada",
    "France",
    "Germany",
    "Hong Kong",
    "India",
    "Ireland",
    "Italy",
    "Mexico",
    "Netherlands",
    "New Zealand",
    "Singapore",
    "Spain",
    "Switzerland",
    "UK",
    "USA",
    "Vietnam"
]

Han pasado 7920 horas desde el 1 de enero de este año.


In [7]:
# --- 3. Lista para guardar resultados ---
# Guardaremos los DataFrames de cada sitio aquí
all_jobs_dfs = []

In [8]:
print("Parámetros listos. Iniciaremos scrapers secuenciales y especializados.")

Parámetros listos. Iniciaremos scrapers secuenciales y especializados.


In [ ]:
# --- 1. Scraper: Indeed (El "Caballo de batalla") ---
# Es el más estable y sin límites de solicitudes

print("\n--- Iniciando Scraper: Indeed/Glassdoor ---")
for country_indeed, search_term, hours_old in itertools.product(indeed_glassdoor_countries, search_terms, hours_old_list):
    print(f"Buscando en {country_indeed} por {search_term} de hace {hours_old} horas")
    try:
        indeed_jobs = safe_scrape_jobs(
            site_name=["indeed", "glassdoor"],
            search_term=search_term,
            country_indeed=country_indeed,
            results_wanted=99999,
            hours_old=hours_old
        )
        if indeed_jobs is not None and not indeed_jobs.empty:
            print(f"✅ Se encontraron {len(indeed_jobs)} trabajos.")
            all_jobs_dfs.append(indeed_jobs)
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")


--- Iniciando Scraper: Indeed/Glassdoor ---
Buscando en Australia por Fintech de hace 0 horas


2025-11-27 20:51:58,857 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 1213 trabajos.
Buscando en Australia por Fintech de hace 24 horas
✅ Se encontraron 5 trabajos.
Buscando en Australia por Fintech de hace 48 horas


2025-11-27 20:52:02,276 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 14 trabajos.
Buscando en Australia por Fintech de hace 72 horas


...

✅ Se encontraron 329 trabajos.
Buscando en Australia por Fintech de hace 2592 horas


2025-11-27 21:14:11,433 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 334 trabajos.
Buscando en Australia por Fintech de hace 2616 horas


2025-11-27 21:14:18,993 - ERROR - JobSpy:Glassdoor - Glassdoor: bad response status code: 429


✅ Se encontraron 334 trabajos.
Buscando en Australia por Fintech de hace 2640 horas


In [ ]:
"""print("\n--- Iniciando Scraper: Google ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        google_jobs = safe_scrape_jobs(
            site_name=["google"],
            search_term=search_term,
            google_search_term=f"{search_term}",
            results_wanted=10,
            hours_old=hours_old,
            verbose=2
        )
        if google_jobs is not None and not google_jobs.empty:
            print(f"✅ Se encontraron {len(google_jobs)} trabajos.")
            all_jobs_dfs.append(google_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

'print("\n--- Iniciando Scraper: Google ---")\nfor search_term, hours_old in itertools.product(search_terms, hours_old_list):\n    print(f"Buscando por {search_term} con {hours_old} horas")\n    try:\n        google_jobs = safe_scrape_jobs(\n            site_name=["google"],\n            search_term=search_term,\n            google_search_term=f"{search_term}",\n            results_wanted=10,\n            hours_old=hours_old,\n            verbose=2\n        )\n        if google_jobs is not None and not google_jobs.empty:\n            print(f"✅ Se encontraron {len(google_jobs)} trabajos.")\n            all_jobs_dfs.append(google_jobs)\n        else:\n            print(f"❌ No se encontraron trabajos para {search_term}")    \n    except Exception as e:\n        print(f"❌ Error después de varios intentos: {e}")'

In [ ]:
"""print("\n--- Iniciando Scraper: ZipRecruiter ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        zip_jobs = safe_scrape_jobs(
            site_name=["zip_recruiter"],
            search_term=search_term,
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ Se encontraron {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""


--- Iniciando Scraper: ZipRecruiter ---


NameError: name 'itertools' is not defined

In [ ]:
"""print("\n--- Iniciando Scraper: Bayt, Naukri, BdJobs ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        bayt_jobs = safe_scrape_jobs(
            site_name=["bayt", "naukri", "bdjobs"],
            search_term=f"{search_term}",
            results_wanted=100, 
            hours_old=hours_old,
            verbose=2
        )
        if bayt_jobs is not None and not bayt_jobs.empty:
            print(f"✅ Se encontraron {len(bayt_jobs)} trabajos.")
            all_jobs_dfs.append(bayt_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")"""

2025-11-27 19:54:11,129 - INFO - JobSpy:Bayt - Fetching Bayt jobs page 1
2025-11-27 19:54:11,130 - INFO - JobSpy:Naukri - Naukri scraper initialized
2025-11-27 19:54:11,131 - INFO - JobSpy:Naukri - Scraping page 1 / 5 for search term: Fintech



--- Iniciando Scraper: Bayt, Naukri, BdJobs ---
Buscando por Fintech


2025-11-27 19:54:11,786 - ERROR - JobSpy:Bayt - Bayt: Error fetching jobs - 403 Client Error: Forbidden for url: https://www.bayt.com/en/international/jobs/Fintech-jobs/?page=1
2025-11-27 19:54:11,787 - INFO - JobSpy:Bayt - finished scraping
2025-11-27 19:54:12,325 - ERROR - JobSpy:Naukri - Naukri API response status code 406 - {"message":"recaptcha required","statusCode":406,"validationErrors":[]}
2025-11-27 19:54:12,326 - INFO - JobSpy:Naukri - finished scraping
2025-11-27 19:54:12,329 - INFO - JobSpy:Bayt - Fetching Bayt jobs page 1
2025-11-27 19:54:12,330 - INFO - JobSpy:Naukri - Naukri scraper initialized
2025-11-27 19:54:12,332 - INFO - JobSpy:Naukri - Scraping page 1 / 5 for search term: EdTech


Unexpected error in safe_scrape_jobs: BDJobs.__init__() got an unexpected keyword argument 'user_agent'
❌ Error después de varios intentos: BDJobs.__init__() got an unexpected keyword argument 'user_agent'
Buscando por EdTech


2025-11-27 19:54:12,653 - ERROR - JobSpy:Bayt - Bayt: Error fetching jobs - 403 Client Error: Forbidden for url: https://www.bayt.com/en/international/jobs/EdTech-jobs/?page=1
2025-11-27 19:54:12,654 - INFO - JobSpy:Bayt - finished scraping
2025-11-27 19:54:12,765 - ERROR - JobSpy:Naukri - Naukri API response status code 406 - {"message":"recaptcha required","statusCode":406,"validationErrors":[]}
2025-11-27 19:54:12,765 - INFO - JobSpy:Naukri - finished scraping
2025-11-27 19:54:12,768 - INFO - JobSpy:Bayt - Fetching Bayt jobs page 1
2025-11-27 19:54:12,770 - INFO - JobSpy:Naukri - Naukri scraper initialized
2025-11-27 19:54:12,771 - INFO - JobSpy:Naukri - Scraping page 1 / 5 for search term: Future of Work


Unexpected error in safe_scrape_jobs: BDJobs.__init__() got an unexpected keyword argument 'user_agent'
❌ Error después de varios intentos: BDJobs.__init__() got an unexpected keyword argument 'user_agent'
Buscando por Future of Work


2025-11-27 19:54:13,096 - ERROR - JobSpy:Bayt - Bayt: Error fetching jobs - 403 Client Error: Forbidden for url: https://www.bayt.com/en/international/jobs/Future%20of%20Work-jobs/?page=1
2025-11-27 19:54:13,097 - INFO - JobSpy:Bayt - finished scraping
2025-11-27 19:54:13,207 - ERROR - JobSpy:Naukri - Naukri API response status code 406 - {"message":"recaptcha required","statusCode":406,"validationErrors":[]}
2025-11-27 19:54:13,208 - INFO - JobSpy:Naukri - finished scraping


Unexpected error in safe_scrape_jobs: BDJobs.__init__() got an unexpected keyword argument 'user_agent'
❌ Error después de varios intentos: BDJobs.__init__() got an unexpected keyword argument 'user_agent'


In [ ]:
print("\n--- Iniciando Scraper: Linkedin ---")
for search_term, hours_old in itertools.product(search_terms, hours_old_list):
    print(f"Buscando por {search_term} con {hours_old} horas")
    try:
        linkedin_jobs = safe_scrape_jobs(
            site_name=["linkedin"],
            search_term=f"{search_term}",
            results_wanted=99999,
            linkedin_fetch_description=True,
            hours_old=hours_old,
            verbose=2
        )
        if linkedin_jobs is not None and not linkedin_jobs.empty:
            print(f"✅ Se encontraron {len(linkedin_jobs)} trabajos.")
            all_jobs_dfs.append(linkedin_jobs)
        else:
            print(f"❌ No se encontraron trabajos para {search_term}")    
    except Exception as e:
        print(f"❌ Error después de varios intentos: {e}")

2025-11-27 19:54:13,225 - INFO - JobSpy:LinkedIn - search page: 1 / 10



--- Iniciando Scraper: Linkedin ---
Buscando por Fintech


2025-11-27 19:54:28,791 - INFO - JobSpy:LinkedIn - search page: 2 / 10
2025-11-27 19:54:40,185 - INFO - JobSpy:LinkedIn - search page: 3 / 10
2025-11-27 19:54:52,739 - INFO - JobSpy:LinkedIn - search page: 4 / 10
2025-11-27 19:55:08,112 - INFO - JobSpy:LinkedIn - search page: 5 / 10
2025-11-27 19:55:20,155 - INFO - JobSpy:LinkedIn - search page: 6 / 10
2025-11-27 19:55:35,441 - INFO - JobSpy:LinkedIn - search page: 7 / 10
2025-11-27 19:55:48,678 - INFO - JobSpy:LinkedIn - search page: 8 / 10
2025-11-27 19:56:01,310 - INFO - JobSpy:LinkedIn - search page: 9 / 10
2025-11-27 19:56:17,019 - INFO - JobSpy:LinkedIn - search page: 10 / 10
2025-11-27 19:56:25,468 - INFO - JobSpy:Linkedin - finished scraping
2025-11-27 19:56:28,097 - INFO - JobSpy:LinkedIn - search page: 1 / 10


✅ Se encontraron 100 trabajos.
Buscando por EdTech


2025-11-27 19:56:43,358 - INFO - JobSpy:LinkedIn - search page: 2 / 10
2025-11-27 19:56:55,779 - INFO - JobSpy:LinkedIn - search page: 3 / 10
2025-11-27 19:57:10,561 - INFO - JobSpy:LinkedIn - search page: 4 / 10
2025-11-27 19:57:22,821 - INFO - JobSpy:LinkedIn - search page: 5 / 10
2025-11-27 19:57:38,276 - INFO - JobSpy:LinkedIn - search page: 6 / 10
2025-11-27 19:57:53,975 - INFO - JobSpy:LinkedIn - search page: 7 / 10
2025-11-27 19:58:06,786 - INFO - JobSpy:LinkedIn - search page: 8 / 10
2025-11-27 19:58:20,508 - INFO - JobSpy:LinkedIn - search page: 9 / 10
2025-11-27 19:58:35,370 - INFO - JobSpy:LinkedIn - search page: 10 / 10
2025-11-27 19:58:43,014 - INFO - JobSpy:Linkedin - finished scraping
2025-11-27 19:58:45,811 - INFO - JobSpy:LinkedIn - search page: 1 / 10


✅ Se encontraron 100 trabajos.
Buscando por Future of Work


2025-11-27 19:58:59,582 - INFO - JobSpy:LinkedIn - search page: 2 / 10
2025-11-27 19:59:12,587 - INFO - JobSpy:LinkedIn - search page: 3 / 10
2025-11-27 19:59:23,797 - INFO - JobSpy:LinkedIn - search page: 4 / 10
2025-11-27 19:59:35,093 - INFO - JobSpy:LinkedIn - search page: 5 / 10
2025-11-27 19:59:47,368 - INFO - JobSpy:LinkedIn - search page: 6 / 10
2025-11-27 19:59:59,807 - INFO - JobSpy:LinkedIn - search page: 7 / 10
2025-11-27 20:00:14,843 - INFO - JobSpy:LinkedIn - search page: 8 / 10
2025-11-27 20:00:28,138 - INFO - JobSpy:LinkedIn - search page: 9 / 10
2025-11-27 20:00:39,452 - INFO - JobSpy:LinkedIn - search page: 10 / 10
2025-11-27 20:00:53,539 - INFO - JobSpy:LinkedIn - search page: 11 / 10
2025-11-27 20:00:54,780 - INFO - JobSpy:Linkedin - finished scraping


✅ Se encontraron 100 trabajos.


In [ ]:
print("\n--- Scraping secuencial completado ---")


--- Scraping secuencial completado ---


In [ ]:
# Update your main processing loop:
if all_jobs_dfs:
    process_and_save_jobs(all_jobs_dfs, output_dir)
else:
    print("\nNo jobs were found.")

Looking for .env at: /home/wagner/Documentos/dev-projects/No Country/Market-Scraper/.env
✅ Successfully connected to MongoDB
📊 Database: job_market
📂 Collection: jobs
🔗 Total documents: 14576

📊 Job Processing Summary
✅ New jobs inserted: 299
🔄 Existing jobs updated: 0
⏩ Jobs unchanged (skipped): 0
❌ Errors: 0
📦 Backup saved to: ../data/raw/jobs_20251127_195323/backup_20251127_200204.json
